In [22]:
import joblib
import json
import pandas as pd
import numpy as np

In [24]:
final_model = joblib.load(
    "output/mitsubishi_lightgbm_demand_model.pkl"
)
weekly_demand = pd.read_csv(
    "output/weekly_demand_features.csv"
)
item_features = pd.read_csv(
    "output/item_features.csv"
)
# Load feature configuration
with open(
    "output/mitsubishi_lightgbm_features.json",
    "r"
) as f:
    features = json.load(f)
with open(
    "output/mitsubishi_lightgbm_categorical_features.json",
    "r"
) as f:
    categorical_features = json.load(f)



print("Number of features:", len(features))
print("Features:")
print(features)

print("\nCategorical features:")
print(categorical_features)


Number of features: 69
Features:
['demand_lag_1', 'demand_lag_2', 'demand_lag_3', 'demand_lag_4', 'demand_lag_5', 'demand_lag_6', 'demand_lag_7', 'demand_lag_8', 'demand_lag_9', 'demand_lag_10', 'demand_lag_11', 'demand_lag_12', 'demand_lag_24', 'demand_lag_36', 'demand_lag_48', 'demand_lag_52', 'demand_mean_2w', 'demand_mean_4w', 'demand_mean_6w', 'demand_mean_8w', 'demand_mean_12w', 'demand_mean_26w', 'demand_mean_52w', 'demand_std_2w', 'demand_std_4w', 'demand_std_6w', 'demand_std_8w', 'demand_std_12w', 'demand_std_26w', 'demand_std_52w', 'demand_trend_2w', 'demand_trend_4w', 'demand_trend_8w', 'demand_trend_12w', 'zero_demand_2w', 'zero_demand_4w', 'zero_demand_6w', 'zero_demand_12w', 'last_nonzero_demand', 'weeks_since_demand', 'yoy_demand_change', 'orders', 'customers', 'production_status', 'product_category_1', 'product_category_2', 'total_orders', 'unique_customers', 'monthly_demand_mean', 'monthly_demand_std', 'demand_cv', 'xyz_class', 'abc_class', 'reorder_customers', 'avg_re

In [25]:
future_weeks = pd.date_range(
    start="2025-01-06",
    end="2025-12-29",
    freq="W-MON"
)

print("Number of forecast weeks:", len(future_weeks))

Number of forecast weeks: 52


In [26]:
def create_demand_features(
    df,
    item_features,
    features
):
    
    df = df.sort_values(
        ["item_internal_id", "week"]
    ).copy()

    grouped = df.groupby(
        "item_internal_id"
    )

    # Lag features
    for lag in [1, 2, 3, 4, 8, 12, 26, 52]:
        df[f"demand_lag_{lag}"] = (
            grouped["demand"].shift(lag)
        )

    # Rolling features
    for window in [4, 8, 12, 26, 52]:

        df[f"demand_mean_{window}w"] = (
            grouped["demand"]
            .shift(1)
            .rolling(window)
            .mean()
            .reset_index(level=0, drop=True)
        )

        df[f"demand_std_{window}w"] = (
            grouped["demand"]
            .shift(1)
            .rolling(window)
            .std()
            .reset_index(level=0, drop=True)
        )

    # Demand trends
    df["demand_trend_4w"] = (
        df["demand_lag_1"] -
        df["demand_lag_4"]
    )

    df["demand_trend_12w"] = (
        df["demand_lag_1"] -
        df["demand_lag_12"]
    )

    # Calendar
    df["month"] = df["week"].dt.month
    df["quarter"] = df["week"].dt.quarter
    df["week_of_year"] = (
        df["week"]
        .dt.isocalendar()
        .week
        .astype(int)
    )
    df["year"] = df["week"].dt.year

    return df

In [27]:
history = weekly_demand[
    weekly_demand["week"] < "2025-01-01"
].copy()

In [28]:
products = history[
    "item_internal_id"
].unique()

future_index = pd.MultiIndex.from_product(
    [products, future_weeks],
    names=[
        "item_internal_id",
        "week"
    ]
)

future = pd.DataFrame(
    index=future_index
).reset_index()

future["demand"] = np.nan

In [29]:
forecast_data = pd.concat(
    [
        history,
        future
    ],
    ignore_index=True
)

forecast_data = forecast_data.sort_values(
    ["item_internal_id", "week"]
).reset_index(drop=True)

In [30]:
forecast_data["week"] = pd.to_datetime(
    forecast_data["week"],
    errors="coerce"
)

print(forecast_data["week"].dtype)
print(forecast_data["week"].isna().sum())

datetime64[us]
0


In [32]:
for forecast_week in future_weeks:

    print(f"Forecasting: {forecast_week.date()}")

    # Recalculate features
    temp = create_demand_features(
        forecast_data,
        item_features,
        features
    )

    # Find the rows for this forecast week
    current_rows = temp["week"] == forecast_week

    # Get features for this week
    X_future = temp.loc[
        current_rows,
        features
    ].copy()

    # Convert categorical columns
    for col in categorical_features:
        X_future[col] = X_future[col].astype("category")

    # Predict
    predictions = final_model.predict(X_future)

    # Demand cannot be negative
    predictions = np.maximum(predictions, 0)

    # Write predictions back
    forecast_data.loc[
        forecast_data["week"] == forecast_week,
        "demand"
    ] = predictions

Forecasting: 2025-01-06
Forecasting: 2025-01-13
Forecasting: 2025-01-20
Forecasting: 2025-01-27
Forecasting: 2025-02-03
Forecasting: 2025-02-10
Forecasting: 2025-02-17
Forecasting: 2025-02-24
Forecasting: 2025-03-03
Forecasting: 2025-03-10
Forecasting: 2025-03-17
Forecasting: 2025-03-24
Forecasting: 2025-03-31
Forecasting: 2025-04-07
Forecasting: 2025-04-14
Forecasting: 2025-04-21
Forecasting: 2025-04-28
Forecasting: 2025-05-05
Forecasting: 2025-05-12
Forecasting: 2025-05-19
Forecasting: 2025-05-26
Forecasting: 2025-06-02
Forecasting: 2025-06-09
Forecasting: 2025-06-16
Forecasting: 2025-06-23
Forecasting: 2025-06-30
Forecasting: 2025-07-07
Forecasting: 2025-07-14
Forecasting: 2025-07-21
Forecasting: 2025-07-28
Forecasting: 2025-08-04
Forecasting: 2025-08-11
Forecasting: 2025-08-18
Forecasting: 2025-08-25
Forecasting: 2025-09-01
Forecasting: 2025-09-08
Forecasting: 2025-09-15
Forecasting: 2025-09-22
Forecasting: 2025-09-29
Forecasting: 2025-10-06
Forecasting: 2025-10-13
Forecasting: 202

In [33]:
forecast_2025 = forecast_data[
    forecast_data["week"].isin(
        future_weeks
    )
].copy()

In [34]:
forecast_2025[
    [
        "item_internal_id",
        "week",
        "demand"
    ]
].rename(
    columns={
        "demand": "predicted_demand"
    }
).to_excel(
    "Mitsubishi_2025_Weekly_Demand_Forecast.xlsx",
    index=False
)

In [35]:
forecast_2025["month"] = (
    forecast_2025["week"]
    .dt.to_period("M")
)

monthly_forecast = (
    forecast_2025
    .groupby(
        [
            "item_internal_id",
            "month"
        ],
        as_index=False
    )["demand"]
    .sum()
    .rename(
        columns={
            "demand": "predicted_demand"
        }
    )
)

monthly_forecast.to_excel(
    "Mitsubishi_2025_Monthly_Demand_Forecast.xlsx",
    index=False
)